# INSPIRE Surgical Copilot

## Notebook 1 – Data Preparation

### Objective

Understand the structure of the INSPIRE dataset before feature engineering or model development.

This notebook focuses on:

- Understanding each table
- Inspecting dataset dimensions
- Reviewing columns
- Assessing missing values
- Understanding table relationships

No preprocessing or machine learning is performed here.

In [1]:
import sys

print("Notebook is working!")
print(sys.executable)

Notebook is working!
c:\Users\Shivani12\Downloads\Masters\NYU\Projects\inspire-surgical-copilot\.venv\Scripts\python.exe


In [2]:
from pathlib import Path

In [3]:
DATA_PATH = Path("../data/raw/inspire")

print(DATA_PATH.resolve())
print(DATA_PATH.exists())

C:\Users\Shivani12\Downloads\Masters\NYU\Projects\inspire-surgical-copilot\data\raw\inspire
True


In [8]:
for file in sorted(DATA_PATH.iterdir()):
    print(file.name)

CHANGELOG.txt
department.csv
diagnosis.csv.gz
icd10_excluded.csv
labs.csv.gz
LICENSE.txt
medications.csv.gz
operations.csv.gz
parameters.csv
schema.csv
SHA256SUMS.txt
vitals.csv.gz
ward_vitals.csv.gz


## Load Operations Table

The operations table is the primary table in the INSPIRE dataset.

Each row represents one surgical procedure and contains patient demographics, surgery details, anesthesia information, and postoperative outcomes.

In [11]:
operations = pd.read_csv(
    DATA_PATH / "operations.csv.gz",
    compression="gzip"
)

In [13]:
print(f"Rows    : {operations.shape[0]:,}")
print(f"Columns : {operations.shape[1]}")

Rows    : 130,960
Columns : 29


In [15]:
operations.head()
operations.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130960 entries, 0 to 130959
Data columns (total 29 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   op_id                130960 non-null  int64  
 1   subject_id           130960 non-null  int64  
 2   hadm_id              130960 non-null  int64  
 3   case_id              21099 non-null   float64
 4   opdate               130960 non-null  int64  
 5   age                  130960 non-null  int64  
 6   sex                  130960 non-null  object 
 7   weight               129493 non-null  float64
 8   height               130172 non-null  float64
 9   race                 130960 non-null  object 
 10  asa                  127413 non-null  float64
 11  emop                 130960 non-null  int64  
 12  department           130960 non-null  object 
 13  antype               130960 non-null  object 
 14  icd10_pcs            130960 non-null  object 
 15  orin_time        

In [17]:
## Operations Table Summary
operations.describe()

,op_id,subject_id,hadm_id,case_id,opdate,age,weight,height,asa,emop,orin_time,orout_time,opstart_time,opend_time,admission_time,discharge_time,anstart_time,anend_time,cpbon_time,cpboff_time,icuin_time,icuout_time,inhosp_death_time,allcause_death_time
count,1.309600e+05,1.309600e+05,1.309600e+05,21099.000000,1.309600e+05,130960.000000,129493.000000,130172.000000,127413.000000,130960.000000,1.309600e+05,1.309600e+05,1.309480e+05,1.309490e+05,1.309600e+05,1.309600e+05,1.309070e+05,1.306050e+05,2.522000e+03,2.522000e+03,1.919500e+04,1.898300e+04,1.555000e+03,2.011400e+04
mean,4.501795e+08,1.501013e+08,2.500675e+08,2652.904640,2.217982e+05,55.740341,62.712309,162.103140,1.762199,0.094151,2.225017e+05,2.226657e+05,2.225498e+05,2.226646e+05,2.182584e+05,2.328001e+05,2.224267e+05,2.226985e+05,1.930163e+05,1.932140e+05,5.928184e+05,5.986553e+05,6.952127e+05,2.066122e+06
std,2.890426e+07,2.885304e+07,2.885166e+07,20961.925401,6.606489e+05,16.042714,12.403815,69.860078,0.638280,0.292040,6.606525e+05,6.606463e+05,6.606727e+05,6.606649e+05,6.604193e+05,6.623017e+05,6.604798e+05,6.607757e+05,6.819669e+05,6.819694e+05,1.130295e+06,1.134157e+06,1.018756e+06,1.580973e+06
min,4.000005e+08,1.000008e+08,2.000009e+08,-32763.000000,-1.440000e+03,20.000000,0.000000,0.000000,1.000000,0.000000,-2.000000e+01,6.000000e+01,0.000000e+00,5.500000e+01,0.000000e+00,1.435000e+03,-1.000000e+01,6.500000e+01,1.150000e+02,2.900000e+02,1.150000e+02,6.100000e+02,7.850000e+02,4.320000e+03
25%,4.251103e+08,1.251247e+08,2.251018e+08,-19802.500000,1.440000e+03,45.000000,55.000000,155.000000,1.000000,0.000000,2.070000e+03,2.210000e+03,2.100000e+03,2.205000e+03,0.000000e+00,5.755000e+03,2.075000e+03,2.210000e+03,4.900000e+03,5.065000e+03,3.790000e+03,5.200000e+03,5.005750e+04,7.286400e+05
50%,4.503241e+08,1.502111e+08,2.501327e+08,8161.000000,2.880000e+03,60.000000,60.000000,160.000000,2.000000,0.000000,3.340000e+03,3.490000e+03,3.390000e+03,3.480000e+03,0.000000e+00,1.151500e+04,3.355000e+03,3.485000e+03,7.800000e+03,8.000000e+03,9.820000e+03,1.387500e+04,1.840250e+05,1.717200e+06
75%,4.752572e+08,1.750137e+08,2.751014e+08,20446.500000,1.008000e+04,70.000000,70.000000,170.000000,2.000000,0.000000,1.065000e+04,1.075000e+04,1.067500e+04,1.074500e+04,0.000000e+00,3.167500e+04,1.065500e+04,1.074000e+04,1.268000e+04,1.282875e+04,6.417650e+05,6.516950e+05,9.459150e+05,3.168000e+06
max,4.999990e+08,1.999994e+08,2.999994e+08,32767.000000,5.184000e+06,90.000000,455.000000,17935.000000,6.000000,1.000000,5.185065e+06,5.185170e+06,5.185090e+06,5.185165e+06,5.182560e+06,5.378395e+06,5.185070e+06,5.185170e+06,5.133215e+06,5.133375e+06,6.096550e+06,6.097965e+06,4.623645e+06,7.470720e+06


In [24]:
def summarize_dataframe(df):
    """Generate a quick overview of a dataframe."""

    summary = pd.DataFrame({
        "Data Type": df.dtypes,
        "Missing Values": df.isnull().sum(),
        "Missing (%)": (df.isnull().mean() * 100).round(2),
        "Unique Values": df.nunique()
    })

    return summary.sort_values("Missing (%)", ascending=False)

summary = summarize_dataframe(operations)

summary


,Data Type,Missing Values,Missing (%),Unique Values
inhosp_death_time,float64,129405,98.81,925
cpboff_time,float64,128438,98.07,1432
cpbon_time,float64,128438,98.07,1301
icuout_time,float64,111977,85.50,7011
icuin_time,float64,111765,85.34,7253
allcause_death_time,float64,110846,84.64,3630
case_id,float64,109861,83.89,21099
asa,float64,3547,2.71,6
weight,float64,1467,1.12,39
height,float64,788,0.60,45


In [25]:
summary.to_csv("../docs/operations_summary.csv", index=True)

In [27]:
operations_clean = operations.copy()
operations_clean.isnull().sum().sort_values(ascending=False)

inhosp_death_time      129405
cpboff_time            128438
cpbon_time             128438
icuout_time            111977
icuin_time             111765
allcause_death_time    110846
case_id                109861
asa                      3547
weight                   1467
height                    788
anend_time                355
anstart_time               53
opstart_time               12
opend_time                 11
department                  0
race                        0
age                         0
sex                         0
subject_id                  0
hadm_id                     0
opdate                      0
op_id                       0
emop                        0
discharge_time              0
admission_time              0
icd10_pcs                   0
antype                      0
orin_time                   0
orout_time                  0
dtype: int64

In [28]:
operations_clean.dtypes

op_id                    int64
subject_id               int64
hadm_id                  int64
case_id                float64
opdate                   int64
age                      int64
sex                     object
weight                 float64
height                 float64
race                    object
asa                    float64
emop                     int64
department              object
antype                  object
icd10_pcs               object
orin_time                int64
orout_time               int64
opstart_time           float64
opend_time             float64
admission_time           int64
discharge_time           int64
anstart_time           float64
anend_time             float64
cpbon_time             float64
cpboff_time            float64
icuin_time             float64
icuout_time            float64
inhosp_death_time      float64
allcause_death_time    float64
dtype: object

In [29]:
operations_clean.columns.tolist()

['op_id',
 'subject_id',
 'hadm_id',
 'case_id',
 'opdate',
 'age',
 'sex',
 'weight',
 'height',
 'race',
 'asa',
 'emop',
 'department',
 'antype',
 'icd10_pcs',
 'orin_time',
 'orout_time',
 'opstart_time',
 'opend_time',
 'admission_time',
 'discharge_time',
 'anstart_time',
 'anend_time',
 'cpbon_time',
 'cpboff_time',
 'icuin_time',
 'icuout_time',
 'inhosp_death_time',
 'allcause_death_time']

In [30]:
operations_clean = operations.copy()
operations_clean["icu_admission"] = (
    operations_clean["icuin_time"]
    .notna()
    .astype(int)
)

In [31]:
model_data = operations_clean[
    [
        "age",
        "sex",
        "weight",
        "height",
        "race",
        "asa",
        "emop",
        "department",
        "antype",
        "icd10_pcs",
        "icu_admission",
    ]
].copy()

In [32]:
model_data["bmi"] = (
    model_data["weight"]
    / (model_data["height"] / 100) ** 2
)

In [33]:
model_data.drop(
    columns=["height", "weight"],
    inplace=True,
)

In [34]:
model_data.to_parquet(
    "../data/processed/model_data.parquet",
    index=False,
)

In [ ]:
from pathlib import Path

Path("../data/processed/model_data.parquet").exists()